In [1]:
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import sqlalchemy

In [2]:
data_path = Path(os.environ["DATA_PATH"])
initial_path = data_path / "initial"
generated_path = data_path / "generated"

population_grids_path = Path(os.environ["POPULATION_GRIDS_PATH"])

In [3]:
engine = sqlalchemy.create_engine(
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}",
)

# Geometría

In [4]:
df_geom = (
    gpd.read_file(initial_path / "r_02a_zonas", columns=["CVEGEO", "_zonaurban"])
    .rename(columns={"_zonaurban": "zone"})
    .query("zone.notna()")
    .assign(zone=lambda df: df["zone"].astype(int).sub(1).astype("category"))
    .set_index("CVEGEO")
    .to_crs("EPSG:6372")
)

# Ingreso

In [5]:
income = gpd.read_file(initial_path / "income.gpkg").set_index("cvegeo")["income_pc"]

# Área baldía

In [6]:
df_barren = (
    gpd.read_file(initial_path / "baldios")
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)
barren_area_frac = (
    df_geom.reset_index()
    .overlay(df_barren, how="intersection")
    .assign(area=lambda df: df["geometry"].area)
    .groupby("CVEGEO")["area"]
    .sum()
    .div(df_geom["geometry"].area)
)

# Delitos

In [7]:
crimes = gpd.read_file(initial_path / "num_delitos_ageb").set_index("CVEGEO")[
    "num_delito"
]

# Valor catastral

In [8]:
df_cat = (
    gpd.read_file(
        initial_path / "valor_cat_actualizado",
        columns=["secsub", "valor2024"],
    )
    .dropna(subset=["secsub"])
    .drop(columns=["secsub"])
    .assign(geometry=lambda df: df["geometry"].force_2d())
    .to_crs("EPSG:6372")
)

In [13]:
with engine.connect() as conn:
    df_census = pd.read_sql(
        """
        SELECT
            cvegeo,
            pobtot,
            p_60ymas,
            p_0a2,
            p_3a5,
            p_6a11,
            p_12a14,
            p_15a17,
            p_18a24,
            pob15_64,
            tvivpar,
            tvivparhab,
            graproes
        FROM census_2020_ageb
        WHERE cvegeo IN %(cvegeos)s
        """,
        conn,
        params={"cvegeos": tuple(df_geom.index)},
    ).set_index("cvegeo")

df_census.columns = df_census.columns.str.upper()

# Cobertura de árboles

In [15]:
with engine.connect() as conn:
    trees = pd.read_sql(
        """
        SELECT cvegeo, area_m2
        FROM tree_coverage_2020_ageb
        WHERE cvegeo IN %(cvegeos)s
        """,
        conn,
        params={"cvegeos": tuple(df_geom.index)},
    ).set_index("cvegeo")["area_m2"]

# Área construida

In [19]:
with engine.connect() as conn:
    urbanized_area = pd.read_sql(
        """
        SELECT cvegeo, area_m2
        FROM urbanized_area_2020_ageb
        WHERE cvegeo IN %(cvegeos)s
        """,
        conn,
        params={"cvegeos": tuple(df_geom.index)},
    ).set_index("cvegeo")["area_m2"]

# Final

In [20]:
df_agebs = (
    df_geom.join(df_census, how="inner")
    .assign(
        income=income,
        vacant_area_frac=barren_area_frac,
        crimes=crimes,
        area_km2=lambda df: df["geometry"].area.div(1e6),
        tree_cover_m2=trees,
        tree_cover_frac=lambda df: df["tree_cover_m2"].div(df["geometry"].area),
        urbanized_area_m2=urbanized_area,
        urbanized_area_frac=lambda df: df["urbanized_area_m2"].div(df["geometry"].area),
    )
    .sort_index()
)
df_agebs.index = df_agebs.index.rename("CVEGEO")
df_agebs.to_file(generated_path / "agebs.gpkg")